Renad Ebn Alameer
renadebnalameer@gmail.com

Saudi Retail Digital Adoption & Performance Dashboard


***الهدف من المشروع:**

تزويد الإدارة العليا ومديري الفروع برؤية شاملة لمسار التحول الرقمي عبر فروع المملكة، من خلال تتبع نمو المبيعات الإلكترونية مقابل الميدانية شهريًا، وتحديد الفروع/المدن الأبطأ في التبني الرقمي، مع مراقبة تأثير هذا التحول على رضا العملاء ومعدلات الإرجاع.




In [47]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv("retail_sales.csv", parse_dates=["date"])
df.head()

,date,store,city,category,channel,units_sold,revenue_sar,returns_units,foot_traffic,csat
0,2025-01-01,Riyadh-Olaya,Riyadh,Electronics,In-Store,52,45097.25,1,950.0,3.98
1,2025-01-01,Riyadh-Olaya,Riyadh,Electronics,Online,12,9323.66,0,NaN,4.00
2,2025-01-01,Riyadh-Olaya,Riyadh,Apparel,In-Store,57,7770.91,2,950.0,4.37
3,2025-01-01,Riyadh-Olaya,Riyadh,Apparel,Online,7,909.04,0,NaN,4.03
4,2025-01-01,Riyadh-Olaya,Riyadh,Home & Kitchen,In-Store,52,11611.75,2,950.0,3.75


In [48]:
import pandas as pd

# 1. تحميل البيانات وتحويل عمود التاريخ
df = pd.read_csv("retail_sales.csv")
df["date"] = pd.to_datetime(df["date"])

# تحويل التاريخ إلى صيغة شهرية (Year-Month)
df["month"] = df["date"].dt.to_period("M")

# تحديد آخر شهر وأحدث 12 شهر في البيانات
latest_month = df["month"].max()
last12 = df[df["date"] > (df["date"].max() - pd.DateOffset(months=12))]
latest_data = df[df["month"] == latest_month]

# --- الأرقام الأساسية اللي بتظهر باللوحة ---

# 1) نسبة التبني الرقمي / المبيعات الإلكترونية (آخر شهر)
online_rev = latest_data.loc[latest_data["channel"] == "Online", "revenue_sar"].sum()
total_rev = latest_data["revenue_sar"].sum()
online_adoption_pct = round((online_rev / total_rev) * 100, 1)

# 2) عدد المدن تحت هدف رضا العملاء (Target CSAT = 4.0)
TARGET_CSAT = 4.0
by_city_latest = latest_data.groupby("city", as_index=False)["csat"].mean()
cities_below_target = int((by_city_latest["csat"] < TARGET_CSAT).sum())

# 3) إجمالي الإيرادات (آخر 12 شهر) — سميته بشكل واضح عشان ما ينكتب فوقه بخطأ بخلية ثانية لاحقًا
total_revenue_12mo = int(last12["revenue_sar"].sum())

# 4) متوسط رضا العملاء (CSAT) لآخر شهر
avg_csat_latest = round(latest_data["csat"].mean(), 2)

# 5) معدل الإرجاع (آخر شهر) — مؤشر جديد يخدم هدفنا الجديد "صحة الأداء"
return_rate_latest = round(
    (latest_data["returns_units"].sum() / latest_data["units_sold"].sum()) * 100, 2
)

# --- طباعة النتائج ---
print(f"Online sales adoption: {online_adoption_pct}%")
print(f"Cities below CSAT target ({TARGET_CSAT}): {cities_below_target}")
print(f"Total revenue (12mo): {total_revenue_12mo:,} SAR")
print(f"Average CSAT (latest month): {avg_csat_latest}")
print(f"Return rate (latest month): {return_rate_latest}%")

Online sales adoption: 26.7%
Cities below CSAT target (4.0): 4
Total revenue (12mo): 148,791,246 SAR
Average CSAT (latest month): 3.91
Return rate (latest month): 3.46%


In [49]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ملاحظة: df, latest_month, latest_data, والمؤشرات (online_adoption_pct, avg_csat_latest,
# return_rate_latest, cities_below_target, total_revenue_12mo) موجودين من الخلية اللي قبل —
# ما نعيد حسابهم هنا عشان ما نطغى فوق قيمهم بمنطق مختلف

# بناء الهيكل الفارغ (Skeleton) — 5 أعمدة عشان نطلع 5 بطاقات KPI بالصف الأول
fig = make_subplots(
    rows=3,
    cols=5,
    specs=[
        [
            {"type": "domain"},
            {"type": "domain"},
            {"type": "domain"},
            {"type": "domain"},
            {"type": "domain"},
        ],
        [{"type": "xy", "colspan": 2}, None, None, {"type": "xy", "colspan": 2}, None],
        [{"type": "xy", "colspan": 5}, None, None, None, None],
    ],
    row_heights=[0.18, 0.42, 0.40],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
    subplot_titles=(
        None,
        None,
        None,
        None,
        None,
        "National digital adoption trend (last 12 months)",
        "Bottom 5 stores — digital adoption (latest month)",
        "Units Sold by Channel",
    ),
)

# ضبط العنوان وأبعاد اللوحة الفارغة
fig.update_layout(height=680, width=1150, title="Empty layout — just the skeleton")

# عرض الشبكة الفارغة
fig.show()

In [50]:
# 4.1 — إضافة بطاقات KPI (الصف الأول)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=online_adoption_pct,
        number={"suffix": "%", "font": {"size": 28}},
        title={"text": "National Digital Adoption"},
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=avg_csat_latest,
        number={"font": {"size": 28}},
        title={"text": "Avg. CSAT (latest month)"},
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=return_rate_latest,
        number={
            "suffix": "%",
            "font": {
                "size": 28,
                "color": "#D55E00" if return_rate_latest > 3 else "#333",
            },
        },
        title={"text": "Return Rate (latest month)"},
    ),
    row=1,
    col=3,
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=cities_below_target,
        number={
            "font": {
                "size": 28,
                "color": "#D55E00" if cities_below_target else "#333",
            }
        },
        title={"text": "Cities Below CSAT Target"},
    ),
    row=1,
    col=4,
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=total_revenue_12mo,
        number={"valueformat": ",.0f", "font": {"size": 28}},
        title={"text": "Total Revenue (12mo)"},
    ),
    row=1,
    col=5,
)

# تحديث العنوان وارتفاع الشكل للخطوة 4.1
fig.update_layout(height=680, width=1150, title="Step 4.1 — KPI cards added")

# عرض اللوحة بعد إضافة بطاقات الـ KPI
fig.show()

In [51]:
# 4.2 — تجهيز بيانات الاتجاه الشهري (National Trend)
# نلخص نسبة المبيعات الرقمية (Online %) لكل شهر
totals_by_month = df.groupby("month")["revenue_sar"].sum()
online_by_month = df.loc[df["channel"] == "Online"].groupby("month")["revenue_sar"].sum()

national_trend = (online_by_month / totals_by_month * 100).reset_index(name="digital_adoption_pct")

# تحويل صيغة الشهر إلى نص متوافق مع محور السينات في Plotly
national_trend["month_str"] = national_trend["month"].astype(str)

# إضافة الرسم الخطي في الصف الثاني - العمود الأول (col=1، colspan=2)
fig.add_trace(
    go.Scatter(
        x=national_trend["month_str"],
        y=national_trend["digital_adoption_pct"],
        mode="lines+markers",
        line=dict(color="#1D9E75", width=2.5),
        showlegend=False,
    ),
    row=2,
    col=1,
)

# تحديث تسميات المحاور
fig.update_xaxes(title_text="Month", row=2, col=1)
fig.update_yaxes(title_text="Adoption (%)", row=2, col=1)

# تحديث عنوان اللوحة والتأكد
fig.update_layout(title="Step 4.2 — Trend line added")
fig.show()

In [52]:
# 4.3 — تجهيز بيانات أقل الفروع أداءً رقميًا (الشهر الأخير)
totals_by_store = latest_data.groupby("store")["revenue_sar"].sum()
online_by_store = latest_data.loc[latest_data["channel"] == "Online"].groupby("store")["revenue_sar"].sum()

by_store_latest = (online_by_store / totals_by_store * 100).reset_index(name="digital_adoption_pct")

# ترتيب البيانات وتحديد أقل 5 فروع
bottom5_stores = by_store_latest.sort_values("digital_adoption_pct").head(5)

# إضافة الشريط الأفقي في الصف الثاني - العمود الرابع (col=4، colspan=2)
fig.add_trace(
    go.Bar(
        x=bottom5_stores["digital_adoption_pct"],
        y=bottom5_stores["store"],
        orientation="h",
        marker_color="#D55E00",
        showlegend=False,
    ),
    row=2,
    col=4,
)

# تحديث تسمية المحور
fig.update_xaxes(title_text="Adoption (%)", row=2, col=4)

# تحديث عنوان اللوحة والتأكد
fig.update_layout(title="Step 4.3 — Bottom 5 stores added")
fig.show()

In [53]:
# 4.4 — تجهيز بيانات توزيع القنوات (الصف الثالث - عرض كامل)
# حساب إجمالي الوحدات المباعة (units_sold) لكل قناة — على مدار السنة كاملة
by_channel = (
    df.groupby("channel", as_index=False)["units_sold"]
    .sum()
    .sort_values("units_sold")
)

# إضافة الرسم البياني للأعمدة الأفقية
fig.add_trace(
    go.Bar(
        x=by_channel["units_sold"],
        y=by_channel["channel"],
        orientation="h",
        marker_color="#378ADD",
        showlegend=False,
    ),
    row=3,
    col=1,
)

# تحديث تسمية المحور السيني
fig.update_xaxes(title_text="Units Sold", row=3, col=1)

# تحديث العنوان النهائي وإظهار اللوحة المكتملة
fig.update_layout(
    title="Step 4.4 — Channel breakdown added — dashboard complete"
)

fig.show()

In [54]:
# 5 — تنظيف العنوان والتنسيق العام (اللمسات النهائية)

fig.update_layout(
    title=dict(
        text="Saudi Retail — Digital Adoption & Performance Dashboard",
        font=dict(size=20, family="Arial"),
        x=0.5,
        xanchor="center",
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial", size=12),
    height=700,
    width=1150,
    margin=dict(l=40, r=40, t=90, b=40),
)

# تعديل حجم خط العناوين الفرعية (Subplot Titles)
for ann in fig.layout.annotations:
    ann.font = dict(size=13)

# عرض اللوحة بالشكل والتنسيق النهائي المكتمل
fig.show()

In [55]:
fig.write_html("retail_sales_dashboard.html", include_plotlyjs="cdn")
print("Saved: retail_sales_dashboard.html")
print("Open this file in any browser — no Python needed to view it.")

Saved: retail_sales_dashboard.html
Open this file in any browser — no Python needed to view it.
